In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
mhs = pd.read_csv("../data/processed/mhs_main_experiment_annotations.csv")

print("Loaded shape:", mhs.shape)
print("Unique comments:", mhs["comment_id"].nunique())
print("Unique annotators:", mhs["annotator_id"].nunique())

Loaded shape: (49433, 52)
Unique comments: 19761
Unique annotators: 7851


In [3]:
survey_item_cols = [
    "sentiment",
    "respect",
    "insult",
    "humiliate",
    "status",
    "dehumanize",
    "violence",
    "genocide",
    "attack_defend",
    "hatespeech"
]

In [4]:
comment_level = (
    mhs.groupby("comment_id")
    .agg(
        text_clean=("text_clean", "first"),
        target_type=("target_type", "first"),
        annotation_count=("annotator_id", "count"),
        unique_annotators=("annotator_id", "nunique"),
        **{f"mean_{col}": (col, "mean") for col in survey_item_cols}
    )
    .reset_index()
)

print("Comment-level shape:", comment_level.shape)
comment_level.head()

Comment-level shape: (19761, 15)


,comment_id,text_clean,target_type,annotation_count,unique_annotators,mean_sentiment,mean_respect,mean_insult,mean_humiliate,mean_status,mean_dehumanize,mean_violence,mean_genocide,mean_attack_defend,mean_hatespeech
0,6,First off you look cool as fuck! Anyway if we ...,women_only,2,2,3.500000,4.000000,3.500000,4.000000,3.000000,3.500000,2.500000,1.000000,3.500000,0.000000
1,7,\*points to posters asking for palestinian rig...,immigrant_only,2,2,3.500000,3.500000,1.500000,1.000000,2.500000,1.500000,2.000000,2.000000,3.500000,1.000000
2,10,"They'll come back in your plan, also. Plus we ...",immigrant_only,3,3,3.333333,3.666667,3.333333,2.666667,3.333333,2.666667,3.666667,3.333333,3.666667,1.333333
3,11,"eat my fuck, bitch",women_only,2,2,4.000000,4.000000,3.000000,2.500000,2.000000,0.500000,0.500000,0.000000,3.000000,1.500000
4,12,She ugly anyways,women_only,2,2,3.500000,4.000000,3.500000,3.000000,2.500000,1.500000,0.500000,0.000000,3.000000,1.000000


In [5]:
mean_cols = [f"mean_{col}" for col in survey_item_cols]

comment_level["composite_severity"] = comment_level[mean_cols].mean(axis=1)

comment_level["composite_bucket"] = pd.qcut(
    comment_level["composite_severity"],
    q=3,
    labels=["low", "medium", "high"],
    duplicates="drop"
)

comment_level["composite_bucket"].value_counts()

composite_bucket
low       6939
medium    6698
high      6124
Name: count, dtype: int64

In [7]:
def final_stratify_label(row):
    if row["target_type"] == "women_and_immigrant":
        return "women_and_immigrant"
    return f"{row['target_type']}__{row['composite_bucket']}"

comment_level["stratify_label"] = comment_level.apply(final_stratify_label, axis=1)

comment_level["stratify_label"].value_counts().sort_index()

stratify_label
immigrant_only__high      2229
immigrant_only__low       3155
immigrant_only__medium    2619
women_and_immigrant        384
women_only__high          3758
women_only__low           3634
women_only__medium        3982
Name: count, dtype: int64

In [8]:
train_comments, temp_comments = train_test_split(
    comment_level,
    test_size=0.30,
    random_state=42,
    stratify=comment_level["stratify_label"]
)

val_comments, test_comments = train_test_split(
    temp_comments,
    test_size=0.50,
    random_state=42,
    stratify=temp_comments["stratify_label"]
)

print("Train comments:", len(train_comments))
print("Validation comments:", len(val_comments))
print("Test comments:", len(test_comments))

Train comments: 13832
Validation comments: 2964
Test comments: 2965


In [10]:
train_comment_ids = set(train_comments["comment_id"])
val_comment_ids = set(val_comments["comment_id"])
test_comment_ids = set(test_comments["comment_id"])

print("Train and Validation:", len(train_comment_ids.intersection(val_comment_ids)))
print("Train and Test:", len(train_comment_ids.intersection(test_comment_ids)))
print("Validation and Test:", len(val_comment_ids.intersection(test_comment_ids)))

Train and Validation: 0
Train and Test: 0
Validation and Test: 0


In [11]:
def assign_split(comment_id):
    if comment_id in train_comment_ids:
        return "train"
    elif comment_id in val_comment_ids:
        return "validation"
    elif comment_id in test_comment_ids:
        return "test"
    return "unknown"

mhs["split"] = mhs["comment_id"].apply(assign_split)

mhs["split"].value_counts()

split
train         34413
test           7713
validation     7307
Name: count, dtype: int64

In [12]:
split_summary = (
    mhs.groupby("split")
    .agg(
        annotation_rows=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        unique_annotators=("annotator_id", "nunique")
    )
    .reset_index()
)

split_summary

,split,annotation_rows,unique_comments,unique_annotators
0,test,7713,2965,4871
1,train,34413,13832,7697
2,validation,7307,2964,4608


In [13]:
strat_total = comment_level.groupby("stratify_label").size().reset_index(name="total_comments")
strat_train = train_comments.groupby("stratify_label").size().reset_index(name="train_comments")
strat_val = val_comments.groupby("stratify_label").size().reset_index(name="validation_comments")
strat_test = test_comments.groupby("stratify_label").size().reset_index(name="test_comments")

strat_balance = (
    strat_total
    .merge(strat_train, on="stratify_label", how="left")
    .merge(strat_val, on="stratify_label", how="left")
    .merge(strat_test, on="stratify_label", how="left")
    .fillna(0)
)

strat_balance[["train_comments", "validation_comments", "test_comments"]] = strat_balance[
    ["train_comments", "validation_comments", "test_comments"]
].astype(int)

strat_balance

,stratify_label,total_comments,train_comments,validation_comments,test_comments
0,immigrant_only__high,2229,1560,335,334
1,immigrant_only__low,3155,2208,473,474
2,immigrant_only__medium,2619,1833,393,393
3,women_and_immigrant,384,269,58,57
4,women_only__high,3758,2631,563,564
5,women_only__low,3634,2544,545,545
6,women_only__medium,3982,2787,597,598


In [14]:
target_split_balance = (
    mhs.groupby(["split", "target_type"])
    .agg(
        annotation_rows=("comment_id", "count"),
        unique_comments=("comment_id", "nunique")
    )
    .reset_index()
)

target_split_balance

,split,target_type,annotation_rows,unique_comments
0,test,immigrant_only,2942,1234
1,test,women_and_immigrant,123,96
2,test,women_only,4648,1742
3,train,immigrant_only,15289,5747
4,train,women_and_immigrant,622,421
5,train,women_only,18502,8117
6,validation,immigrant_only,3313,1241
7,validation,women_and_immigrant,239,92
8,validation,women_only,3755,1737


In [15]:
women_gender_balance = (
    mhs[mhs["is_women_targeted"] == 1]
    .groupby(["split", "annotator_gender_group"])
    .size()
    .reset_index(name="annotation_rows")
)

women_gender_balance

,split,annotator_gender_group,annotation_rows
0,test,men,2021
1,test,non_binary,28
2,test,prefer_not_to_say,16
3,test,self_describe,8
4,test,women,2698
5,train,men,8145
6,train,non_binary,158
7,train,prefer_not_to_say,65
8,train,self_describe,21
9,train,women,10735


In [16]:
immigrant_ideology_balance = (
    mhs[mhs["is_immigrant_targeted"] == 1]
    .groupby(["split", "annotator_ideology_group"])
    .size()
    .reset_index(name="annotation_rows")
)

immigrant_ideology_balance

,split,annotator_ideology_group,annotation_rows
0,test,conservative,324
1,test,extremely_conservative,103
2,test,extremely_liberal,419
3,test,liberal,753
4,test,neutral,559
5,test,no_opinion,90
6,test,slightly_conservative,321
7,test,slightly_liberal,494
8,test,unknown,2
9,train,conservative,1755


In [17]:
survey_split_summary = (
    mhs.groupby("split")[survey_item_cols]
    .mean()
    .reset_index()
)

survey_split_summary

,split,sentiment,respect,insult,humiliate,status,dehumanize,violence,genocide,attack_defend,hatespeech
0,test,2.994814,2.780889,2.492545,2.241800,2.702321,1.835473,0.902762,0.430312,2.483469,0.639440
1,train,3.075001,2.981199,2.707814,2.422980,2.749513,1.985790,1.149624,0.662046,2.709354,0.721878
2,validation,3.163268,3.103052,2.841796,2.541262,2.814014,2.078144,0.920761,0.462570,2.792938,0.767483


In [18]:
comment_split_summary = (
    comment_level.assign(
        split=comment_level["comment_id"].apply(assign_split)
    )
    .groupby("split")
    .agg(
        unique_comments=("comment_id", "count"),
        mean_annotation_count=("annotation_count", "mean"),
        median_annotation_count=("annotation_count", "median"),
        mean_composite_severity=("composite_severity", "mean"),
        min_composite_severity=("composite_severity", "min"),
        max_composite_severity=("composite_severity", "max")
    )
    .reset_index()
)

comment_split_summary

,split,unique_comments,mean_annotation_count,median_annotation_count,mean_composite_severity,min_composite_severity,max_composite_severity
0,test,2965,2.601349,2.0,1.976710,0.0,3.8
1,train,13832,2.487927,2.0,1.966108,0.0,3.8
2,validation,2964,2.465250,2.0,1.972366,0.0,3.8


In [19]:
os.makedirs("../data/processed", exist_ok=True)

train_df = mhs[mhs["split"] == "train"].copy()
val_df = mhs[mhs["split"] == "validation"].copy()
test_df = mhs[mhs["split"] == "test"].copy()

train_df.to_csv("../data/processed/train_annotations.csv", index=False)
val_df.to_csv("../data/processed/validation_annotations.csv", index=False)
test_df.to_csv("../data/processed/test_annotations.csv", index=False)

mhs.to_csv("../data/processed/mhs_main_experiment_annotations_with_split.csv", index=False)

print("Saved split datasets.")

Saved split datasets.


In [21]:
women_gender_balance
immigrant_ideology_balance

,split,annotator_ideology_group,annotation_rows
0,test,conservative,324
1,test,extremely_conservative,103
2,test,extremely_liberal,419
3,test,liberal,753
4,test,neutral,559
5,test,no_opinion,90
6,test,slightly_conservative,321
7,test,slightly_liberal,494
8,test,unknown,2
9,train,conservative,1755


In [22]:
women_gender_percentage = (
    women_gender_balance
    .copy()
)

women_gender_percentage["percentage"] = (
    women_gender_percentage.groupby("split")["annotation_rows"]
    .transform(lambda x: (x / x.sum()) * 100)
)

women_gender_percentage["percentage"] = (
    women_gender_percentage["percentage"].round(2)
)

women_gender_percentage

,split,annotator_gender_group,annotation_rows,percentage
0,test,men,2021,42.36
1,test,non_binary,28,0.59
2,test,prefer_not_to_say,16,0.34
3,test,self_describe,8,0.17
4,test,women,2698,56.55
5,train,men,8145,42.59
6,train,non_binary,158,0.83
7,train,prefer_not_to_say,65,0.34
8,train,self_describe,21,0.11
9,train,women,10735,56.13


In [23]:
immigrant_ideology_percentage = (
    immigrant_ideology_balance
    .copy()
)

immigrant_ideology_percentage["percentage"] = (
    immigrant_ideology_percentage.groupby("split")["annotation_rows"]
    .transform(lambda x: (x / x.sum()) * 100)
)

immigrant_ideology_percentage["percentage"] = (
    immigrant_ideology_percentage["percentage"].round(2)
)

immigrant_ideology_percentage

,split,annotator_ideology_group,annotation_rows,percentage
0,test,conservative,324,10.57
1,test,extremely_conservative,103,3.36
2,test,extremely_liberal,419,13.67
3,test,liberal,753,24.57
4,test,neutral,559,18.24
5,test,no_opinion,90,2.94
6,test,slightly_conservative,321,10.47
7,test,slightly_liberal,494,16.12
8,test,unknown,2,0.07
9,train,conservative,1755,11.03


In [24]:
os.makedirs("../results/tables", exist_ok=True)

report_path = "../results/tables/train_val_test_split_summary.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write("TRAIN / VALIDATION / TEST SPLIT SUMMARY\n")
    f.write("=" * 70 + "\n\n")

    f.write("1. SPLIT SUMMARY\n")
    f.write("-" * 70 + "\n")
    f.write(split_summary.to_string(index=False))
    f.write("\n\n")

    f.write("2. STRATIFICATION METHOD\n")
    f.write("-" * 70 + "\n")
    f.write("Split unit: comment_id\n")
    f.write("Primary stratification: target_type + composite_severity_bucket\n")
    f.write("Composite severity: mean of the 10 MHS survey item means per comment\n")
    f.write("women_and_immigrant was kept as a single stratum throughout due to smaller sample size.\n\n")

    f.write("3. STRATIFICATION BALANCE\n")
    f.write("-" * 70 + "\n")
    f.write(strat_balance.to_string(index=False))
    f.write("\n\n")

    f.write("4. TARGET TYPE BALANCE ACROSS SPLITS\n")
    f.write("-" * 70 + "\n")
    f.write(target_split_balance.to_string(index=False))
    f.write("\n\n")

    f.write("5. WOMEN-TARGETED CONTENT: ANNOTATOR GENDER BALANCE\n")
    f.write("-" * 70 + "\n")
    f.write(women_gender_balance.to_string(index=False))
    f.write("\n\n")

    f.write("6. IMMIGRANT-TARGETED CONTENT: ANNOTATOR IDEOLOGY BALANCE\n")
    f.write("-" * 70 + "\n")
    f.write(immigrant_ideology_balance.to_string(index=False))
    f.write("\n\n")

    f.write("7. SURVEY ITEM MEANS BY SPLIT\n")
    f.write("-" * 70 + "\n")
    f.write(survey_split_summary.to_string(index=False))
    f.write("\n\n")

    f.write("8. COMMENT-LEVEL SPLIT SUMMARY\n")
    f.write("-" * 70 + "\n")
    f.write(comment_split_summary.to_string(index=False))
    f.write("\n\n")

    f.write("9. LEAKAGE CHECK\n")
    f.write("-" * 70 + "\n")
    f.write(f"Train and Validation comments: {len(train_comment_ids.intersection(val_comment_ids))}\n")
    f.write(f"Train and Test comments: {len(train_comment_ids.intersection(test_comment_ids))}\n")
    f.write(f"Validation and Test comments: {len(val_comment_ids.intersection(test_comment_ids))}\n")

    f.write("5B. WOMEN-TARGETED CONTENT: GENDER PERCENTAGES\n")
    f.write("-" * 70 + "\n")
    f.write(women_gender_percentage.to_string(index=False))
    f.write("\n\n")

    f.write("6B. IMMIGRANT-TARGETED CONTENT: IDEOLOGY PERCENTAGES\n")
    f.write("-" * 70 + "\n")
    f.write(immigrant_ideology_percentage.to_string(index=False))
    f.write("\n\n")
print("Saved report:", report_path)

Saved report: ../results/tables/train_val_test_split_summary.txt
